# JAWAD HASSAN
# 2230-0035
# BS AI
# ML
# LAB 09
# OPEN ENDED LAB

### Datasets Used:
[credit card fraud](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud)

In [1]:
#libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import RobustScaler
from imblearn.over_sampling import SMOTE
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv1D, Flatten, MaxPooling1D, Input


2026-04-12 23:10:22.668520: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


# Dataset Study:
## Only three features are given: Time, Amount, Class
## Rest 28 are fully anonymized features (V1 through V28) that have been processed using PCA (Principal Component Analysis) to protect user privacy.
## I cannot extract literal account ages or geographic locations directly from the existing columns. To fulfill my assignment requirements, I will need to synthetically generate these new attributes.

In [2]:
# Question 1: Data Understanding and Preprocessing
# Load dataset
df = pd.read_csv('/Users/macbook/Desktop/sem6/ML/lab9/git9/creditcard.csv')
# Set a seed for reproducibility
np.random.seed(42)
# Account Age Category
# Generate random ages between 1 and 3650 days (10 years)
df['account_age_days'] = np.random.randint(1, 3650, size=len(df))
df['account_age_category'] = pd.cut(df['account_age_days'], 
                                    bins=[0, 30, 365, 4000], 
                                    labels=['New', 'Established', 'Veteran'])

# Country Mismatch Indicator (0 = Match, 1 = Mismatch)
# Make it so fraud cases have an 80% chance of a mismatch, and legit have a 5% chance
def generate_mismatch(is_fraud):
    if is_fraud == 1:
        return np.random.choice([0, 1], p=[0.2, 0.8])
    else:
        return np.random.choice([0, 1], p=[0.95, 0.05])

df['country_mismatch_indicator'] = df['Class'].apply(generate_mismatch)

#  Regulatory Risk Flag (0 = Normal, 1 = High Risk)
# Flag any transaction amount over an arbitrary high threshold, e.g., $5,000
df['regulatory_risk_flag'] = np.where(df['Amount'] > 5000, 1, 0)

# Clean up temporary columns and check the new structure
df = df.drop(columns=['account_age_days'])

print("\nFeature engineering complete. Here is a peek at the new columns:")
print(df[['Time', 'Amount', 'Class', 'account_age_category', 'country_mismatch_indicator', 'regulatory_risk_flag']].head())


Feature engineering complete. Here is a peek at the new columns:
   Time  Amount  Class account_age_category  country_mismatch_indicator  \
0   0.0  149.62      0              Veteran                           0   
1   0.0    2.69      0              Veteran                           0   
2   1.0  378.66      0              Veteran                           0   
3   1.0  123.50      0              Veteran                           0   
4   2.0   69.99      0              Veteran                           0   

   regulatory_risk_flag  
0                     0  
1                     0  
2                     0  
3                     0  
4                     0  


In [3]:
#  EXPLORE THE DATASET
print("--- Data Characteristics ---")
print(f"Dataset Shape: {df.shape}")
print(f"Missing Values:\n{df.isnull().sum().max()} (Max missing in any column)")

# Check the exact class imbalance
fraud_count = df['Class'].value_counts()[1]
legit_count = df['Class'].value_counts()[0]
imbalance_ratio = (fraud_count / len(df)) * 100

print(f"\nLegitimate Transactions: {legit_count}")
print(f"Fraudulent Transactions: {fraud_count}")
print(f"Fraud Percentage: {imbalance_ratio:.3f}%\n")

# Scale 'Amount' and 'Time'
# We use RobustScaler because it is less prone to outliers than StandardScaler
robust_scaler = RobustScaler()
df['scaled_amount'] = robust_scaler.fit_transform(df['Amount'].values.reshape(-1,1))
df['scaled_time'] = robust_scaler.fit_transform(df['Time'].values.reshape(-1,1))

# Drop the original unscaled columns and reorganize
df.drop(['Time', 'Amount'], axis=1, inplace=True)
scaled_amount = df['scaled_amount']
scaled_time = df['scaled_time']
df.drop(['scaled_amount', 'scaled_time'], axis=1, inplace=True)
df.insert(0, 'scaled_amount', scaled_amount)
df.insert(1, 'scaled_time', scaled_time)

#TRAIN-TEST SPLITTING
# Separate features (X) and target (y)
# Note: We must encode our synthetic categorical variables before splitting
df = pd.get_dummies(df, columns=['account_age_category'], drop_first=True)

X = df.drop('Class', axis=1)
y = df['Class']

# Perform the Stratified Split (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2, 
    random_state=42, 
    stratify=y  # This guarantees the test set has the same 0.17% fraud ratio
)

print("--- Data Splitting ---")
print(f"Training set size: {len(X_train)}")
print(f"Testing set size: {len(X_test)}\n")


#HANDLE CLASS IMBALANCE (SMOTE)
# Apply SMOTE ONLY to the training data to prevent data leakage
print("--- Handling Class Imbalance ---")
print("Applying SMOTE to training data...")

smote = SMOTE(sampling_strategy='minority', random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("Before SMOTE - Fraud count:", sum(y_train == 1))
print("Before SMOTE - Legit count:", sum(y_train == 0))
print("After SMOTE - Fraud count:", sum(y_train_smote == 1))
print("After SMOTE - Legit count:", sum(y_train_smote == 0))

--- Data Characteristics ---
Dataset Shape: (284807, 34)
Missing Values:
0 (Max missing in any column)

Legitimate Transactions: 284315
Fraudulent Transactions: 492
Fraud Percentage: 0.173%

--- Data Splitting ---
Training set size: 227845
Testing set size: 56962

--- Handling Class Imbalance ---
Applying SMOTE to training data...


/Users/macbook/Desktop/env/lib/python3.9/site-packages/sklearn/base.py:474: FutureWarning: `BaseEstimator._validate_data` is deprecated in 1.6 and will be removed in 1.7. Use `sklearn.utils.validation.validate_data` instead. This function becomes public and is part of the scikit-learn developer API.
  warnings.warn(


Before SMOTE - Fraud count: 394
Before SMOTE - Legit count: 227451
After SMOTE - Fraud count: 227451
After SMOTE - Legit count: 227451


## We handled class imbalance using SMOTE technique,
## Synthetic Minority Over-sampling Technique (SMOTE) generates synthetic examples of the minority class (fraud) by interpolating between existing fraud cases in the training set.

In [5]:
# Question 2: Machine Learning Model Implementation
def evaluate_model(model_name, y_true, y_pred):
    print(f"--- {model_name} Evaluation ---")
    print(f"Accuracy:  {accuracy_score(y_true, y_pred):.4f}")
    print(f"Precision: {precision_score(y_true, y_pred):.4f}")
    print(f"Recall:    {recall_score(y_true, y_pred):.4f}")
    print(f"F1-Score:  {f1_score(y_true, y_pred):.4f}\n")


#  Decision Tree Classifier
print("Training Decision Tree...")
# Using max_depth to prevent the tree from memorizing the training data (overfitting)
dt_classifier = DecisionTreeClassifier(max_depth=10, random_state=42)
dt_classifier.fit(X_train_smote, y_train_smote)

dt_predictions = dt_classifier.predict(X_test)
evaluate_model("Decision Tree", y_test, dt_predictions)

#  Support Vector Machine (SVM)
print("Training Optimized Linear SVM...")

# Option 1: LinearSVC (Much faster than standard SVC)
# dual=False is strictly recommended when n_samples > n_features
svm_classifier = LinearSVC(random_state=42, dual=False)

# Option 2: SGDClassifier (Uncomment the line below if LinearSVC is still too slow)
# svm_classifier = SGDClassifier(loss='hinge', random_state=42, max_iter=1000)

# Fit the model
svm_classifier.fit(X_train_smote, y_train_smote)

# Predict and evaluate
svm_predictions = svm_classifier.predict(X_test)
evaluate_model("Optimized SVM", y_test, svm_predictions)

#  Convolutional Neural Network (CNN)
print("Training CNN (Conceptual 1D)")


#  Explicitly cast the DataFrames to float32 before calling .values
X_train_cnn = X_train_smote.astype('float32').values.reshape(X_train_smote.shape[0], X_train_smote.shape[1], 1)
X_test_cnn = X_test.astype('float32').values.reshape(X_test.shape[0], X_test.shape[1], 1)

# Cast the target labels to float32 as well just to be safe
y_train_cnn = y_train_smote.astype('float32')

# Build the simplified architecture
cnn_model = Sequential([
    Input(shape=(X_train_smote.shape[1], 1)),
    Conv1D(filters=32, kernel_size=2, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid') 
])

# Compile the model
cnn_model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train the model (using your updated 5 epochs)
cnn_model.fit(X_train_cnn, y_train_cnn, epochs=5, batch_size=64, verbose=1)

# Predict using the CNN
cnn_probs = cnn_model.predict(X_test_cnn)
cnn_predictions = (cnn_probs > 0.5).astype(int).flatten()

evaluate_model("1D Convolutional Neural Network", y_test, cnn_predictions)

Training Decision Tree...
--- Decision Tree Evaluation ---
Accuracy:  0.9907
Precision: 0.1383
Recall:    0.8367
F1-Score:  0.2373

Training Optimized Linear SVM...
--- Optimized SVM Evaluation ---
Accuracy:  0.9801
Precision: 0.0761
Recall:    0.9490
F1-Score:  0.1409

Training CNN (Conceptual 1D)
Epoch 1/5
7108/7108 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.9813 - loss: 0.0515
Epoch 2/5
7108/7108 ━━━━━━━━━━━━━━━━━━━━ 18s 3ms/step - accuracy: 0.9973 - loss: 0.0107
Epoch 3/5
7108/7108 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.9985 - loss: 0.0062
Epoch 4/5
7108/7108 ━━━━━━━━━━━━━━━━━━━━ 20s 3ms/step - accuracy: 0.9988 - loss: 0.0046
Epoch 5/5
7108/7108 ━━━━━━━━━━━━━━━━━━━━ 19s 3ms/step - accuracy: 0.9991 - loss: 0.0043
1781/1781 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step
--- 1D Convolutional Neural Network Evaluation ---
Accuracy:  0.9987
Precision: 0.5929
Recall:    0.8469
F1-Score:  0.6975



## Question 3: Comparative Analysis
### Decision Tree
### Accuracy: 0.9907, Recall: 0.8367 Precision: 0.1383, F1: 0.2373
### High recall but very low precision → many false positives

### Optimized SVM

### Accuracy: 0.9801, Recall: 0.9490 Precision: 0.0761, F1: 0.1409
### Highest recall but worst precision and F1-score

### 1D CNN

### Accuracy: 0.9987, Recall: 0.8469 Precision: 0.5929, F1: 0.6975
### Best balance with highest F1-score

### Final Selection
### CNN is the best model as it has the highest F1-score (0.6975) and much better precision (0.5929) while maintaining high recall (0.8469), making it most suitable for imbalanced data.

In [6]:
# Question 4: Constraint Satisfaction–Based Decision Validation

#  Create a DataFrame for our final analysis
# We use X_test and add the CNN's predicted probabilities
results_df = X_test.copy()
# Flatten cnn_probs to ensure it matches the dataframe index
results_df['ml_fraud_probability'] = cnn_probs.flatten() 

#  Define the Decision Engine Function
def apply_business_rules(row):
    prob = row['ml_fraud_probability']
    
    # Identify 'New' accounts. Because we used drop_first=True when making dummy 
    # variables earlier, 'New' is true when both Established and Veteran are 0.
    is_new_account = (row.get('account_age_category_Established', 0) == 0) and \
                     (row.get('account_age_category_Veteran', 0) == 0)
                     
    mismatch = (row['country_mismatch_indicator'] == 1)
    reg_flag = (row['regulatory_risk_flag'] == 1)
    
    # HARD CONSTRAINTS (Overrides ML Model)
    # Rule 1: High transaction amount (reg_flag) + country mismatch -> Escalate
    if reg_flag and mismatch:
        return pd.Series(['Escalate', 'Hard Rule Triggered: High transaction amount originating from a mismatched country.'])
        
    # Rule 2: Low account age + high transaction amount -> Block
    if is_new_account and reg_flag:
        return pd.Series(['Block', 'Hard Rule Triggered: Very new account attempting a high-value transaction.'])
        
    # Rule 3: Regulatory flag set -> Escalate
    if reg_flag:
        return pd.Series(['Escalate', 'Hard Rule Triggered: Transaction amount exceeds regulatory reporting thresholds.'])


    # SOFT CONSTRAINTS (Relies on ML Model)

    # We define thresholds for what constitutes "high" or "low" probability
    HIGH_PROB_THRESHOLD = 0.85
    LOW_PROB_THRESHOLD = 0.15
    
    # Rule 4: Prefer blocking/escalation for high fraud probability
    if prob >= HIGH_PROB_THRESHOLD:
        return pd.Series(['Block', f'Soft Rule Applied: ML model indicates severe fraud probability ({prob:.2%}).'])
    elif prob >= LOW_PROB_THRESHOLD:
        return pd.Series(['Escalate', f'Soft Rule Applied: ML model indicates suspicious activity ({prob:.2%}). Manual review required.'])
        
    # Rule 5: Prefer approval for low fraud probability
    else:
        return pd.Series(['Approve', f'Soft Rule Applied: ML model indicates low fraud probability ({prob:.2%}). Normal transaction.'])

#  Apply the rules engine to the dataset
print("Applying constraint-based decision model...")
results_df[['final_decision', 'justification']] = results_df.apply(apply_business_rules, axis=1)

#  View the final output
# Let's look at a sample of transactions that were Blocked or Escalated
actioned_transactions = results_df[results_df['final_decision'].isin(['Block', 'Escalate'])]

print("\n--- Decision Engine Output Sample ---")
display_cols = ['ml_fraud_probability', 'final_decision', 'justification']
print(actioned_transactions[display_cols].head(10))

Applying constraint-based decision model...

--- Decision Engine Output Sample ---
        ml_fraud_probability final_decision  \
190263              0.999222          Block   
93527               0.944055          Block   
77348               0.999997          Block   
25717               0.535818       Escalate   
102442              1.000000          Block   
218841              0.473237       Escalate   
54625               0.731265       Escalate   
14920               1.000000          Block   
191439              0.928345          Block   
153457              0.890024          Block   

                                            justification  
190263  Soft Rule Applied: ML model indicates severe f...  
93527   Soft Rule Applied: ML model indicates severe f...  
77348   Soft Rule Applied: ML model indicates severe f...  
25717   Soft Rule Applied: ML model indicates suspicio...  
102442  Soft Rule Applied: ML model indicates severe f...  
218841  Soft Rule Applied: ML model ind

## Justify final decisions using rule-based reasoning:
### Rule-Based Reasoning: By returning a text string right alongside the decision, you have a programmatic audit trail. If a professor (or a real-world auditor) asks why transaction 4592 was blocked, the justification column provides the exact logical path taken.

### Integration: It perfectly marries the "black box" nature of a Neural Network with the "white box" transparency of a rules-based expert system.

### Machine learning models are powerful tools for identifying patterns in large datasets, but they are rarely used in isolation for critical financial decisions. Instead, they are often combined with strict, rule-based expert systems. These rules act as a necessary safety net, ensuring that non-negotiable business logic—like blocking massive transactions from brand-new accounts—always takes precedence over algorithmic probabilities. This hybrid approach provides banks with both the adaptability of artificial intelligence and the rigid transparency required by industry regulators.